# Tech Challenge Fase 3 — Camada Gold: Análise e Visualizações

## State of Data Brasil (2023, 2024 e 2025)

In [ ]:
%idle_timeout 60
%glue_version 4.0
%worker_type G.1X
%number_of_workers 3
%additional_python_modules matplotlib,seaborn,pandas

## 0. Ambiente, leitura da Gold e identidade visual

In [ ]:
import os
import textwrap
from pathlib import Path

from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql import functions as F

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

DATABASE = "state_of_data_db"
TABLE = "state_of_data"

df = spark.sql(f"SELECT * FROM {DATABASE}.{TABLE}")
df.createOrReplaceTempView("state_of_data")


OUTPUT_DIR = Path("graficos_executivos")
OUTPUT_DIR.mkdir(exist_ok=True)

# -------------------------------------------------------------------
# Tema
# -------------------------------------------------------------------
sns.set_theme(style="whitegrid")

PALETTE = {
    "primary": "#3F5E8C",
    "accent": "#8F4E7A",
    "accent2": "#4C8C72",
    "neutral": "#B8C0CC",
    "dark": "#2F3A45",
    "light": "#E9EDF2",
    "female": "#C66B8E",
    "male": "#4F81BD",
    "warning": "#D08C3F",
    "success": "#5C9A6D"
}

plt.rcParams.update({
    "figure.figsize": (12, 7),
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.labelcolor": PALETTE["dark"],
    "xtick.color": PALETTE["dark"],
    "ytick.color": PALETTE["dark"]
})

print(f"Total de registros: {df.count():,}")
df.printSchema()

## Funções de visualização

Todas as seções usam estas funções para manter consistência de fonte, margens, rótulos.

In [ ]:
def fmt_int(v):
    return f"{int(round(v)):,}".replace(",", ".")

def fmt_pct(v, dec=1):
    return f"{v:.{dec}f}%".replace(".", ",")

def wrap_label(text, width=34):
    return "\n".join(textwrap.wrap(str(text), width=width))

def base_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.grid(axis="x", color="#D9DEE7", linewidth=0.8)
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", colors=PALETTE["dark"])
    return ax

def set_exec_title(ax, title, subtitle=None):
    ax.set_title(title, loc="left", fontsize=18, color=PALETTE["dark"], pad=18)
    if subtitle:
        ax.text(
            0, 1.01, subtitle,
            transform=ax.transAxes,
            ha="left", va="bottom",
            fontsize=10.5, color="#667085"
        )

def save_exec(fig, filename):
    fig.tight_layout()
    path = OUTPUT_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"✓ {path}")
    plt.show()

def annotate_hbars(ax, values, x_offset=0.01, suffix="", fmt=fmt_int):
    xmax = max(values) if len(values) else 1
    for i, v in enumerate(values):
        ax.text(
            v + xmax*x_offset, i,
            f"{fmt(v)}{suffix}",
            va="center", ha="left",
            fontsize=10, fontweight="bold",
            color=PALETTE["dark"]
        )

def plot_hbar(
    data, category, value, title, subtitle, filename,
    highlight_top=1, color=PALETTE["primary"], wrap=38, value_fmt=fmt_int
):
    d = data.copy().sort_values(value, ascending=True)
    labels = d[category].map(lambda x: wrap_label(x, wrap))
    colors = [PALETTE["light"]]*len(d)
    for pos in range(min(highlight_top, len(d))):
        colors[-(pos+1)] = color

    fig, ax = plt.subplots(figsize=(12, max(5.5, len(d)*0.58)))
    ax.barh(labels, d[value], color=colors)
    base_axis(ax)
    set_exec_title(ax, title, subtitle)
    annotate_hbars(ax, d[value].tolist(), fmt=value_fmt)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xlim(0, d[value].max()*1.18 if len(d) else 1)
    save_exec(fig, filename)

def plot_100_stacked(pivot_pct, title, subtitle, filename, colors, legend_title, horizontal=False):
    fig, ax = plt.subplots(figsize=(12, 7))
    if horizontal:
        pivot_pct.plot(kind="barh", stacked=True, ax=ax, color=colors, width=0.72)
        ax.set_xlim(0, 100)
        ax.xaxis.set_major_formatter(PercentFormatter(100))
        ax.set_xlabel("% dentro do grupo")
    else:
        pivot_pct.plot(kind="bar", stacked=True, ax=ax, color=colors, width=0.72)
        ax.set_ylim(0, 100)
        ax.yaxis.set_major_formatter(PercentFormatter(100))
        ax.set_ylabel("% de profissionais")
        ax.set_xlabel("")

    set_exec_title(ax, title, subtitle)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#D9DEE7")
    ax.grid(axis="x", visible=False)

    # Rótulos internos
    for container in ax.containers:
        labels=[]
        for bar in container:
            val = bar.get_width() if horizontal else bar.get_height()
            labels.append(fmt_pct(val,0) if val >= 7 else "")
        ax.bar_label(container, labels=labels, label_type="center", fontsize=9, color="white", fontweight="bold")

    ax.legend(title=legend_title, frameon=False, bbox_to_anchor=(1.02,1), loc="upper left")
    save_exec(fig, filename)

def add_last_labels(ax, x, ydf, colors):
    for col, color in zip(ydf.columns, colors):
        y = ydf[col].iloc[-1]
        ax.text(x[-1]+0.03, y, f"{col}: {fmt_pct(y,0)}", va="center", fontsize=9, color=color, fontweight="bold")

def insight_box(fig, title, text):
    fig.text(
        0.02, 0.01,
        f"{title}: {text}",
        ha="left", va="bottom",
        fontsize=10,
        color=PALETTE["dark"],
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#F5F7FA", edgecolor="#D9DEE7")
    )

# 1. Como está estruturado o mercado brasileiro de Dados?

Os gráficos respondem:
1. Quais cargos concentram o maior volume de profissionais em 2025?
2. Como a composição da senioridade evoluiu entre os anos?

In [ ]:
# 1A — Top cargos em 2025
cargos_2025 = spark.sql('''
    SELECT cargo_atual, COUNT(*) AS total
    FROM state_of_data
    WHERE ano_pesquisa = '2025'
      AND cargo_atual IS NOT NULL
      AND TRIM(cargo_atual) != ''
    GROUP BY cargo_atual
    ORDER BY total DESC
    LIMIT 10
''').toPandas()

total_2025 = spark.sql("SELECT COUNT(*) AS total FROM state_of_data WHERE ano_pesquisa='2025'").toPandas().iloc[0,0]
top3_share = cargos_2025.head(3)["total"].sum() / cargos_2025["total"].sum() * 100 if len(cargos_2025) else 0

plot_hbar(
    cargos_2025,
    category="cargo_atual",
    value="total",
    title="Os cargos com maior presença no mercado de Dados em 2025",
    subtitle=f"Top 10 cargos por número de profissionais respondentes | Base 2025: {fmt_int(total_2025)} registros",
    filename="01_top_cargos_2025.png",
    highlight_top=3,
    color=PALETTE["primary"],
    wrap=42
)

print(f"Os 3 principais cargos representam {fmt_pct(top3_share)} do volume exibido no Top 10.")

In [ ]:
# 1B — Evolução da senioridade (percentual para comparabilidade entre anos)
senioridade_ano = spark.sql('''
    SELECT ano_pesquisa, nivel_senioridade, COUNT(*) AS total
    FROM state_of_data
    WHERE nivel_senioridade IS NOT NULL
      AND TRIM(nivel_senioridade) != ''
    GROUP BY ano_pesquisa, nivel_senioridade
''').toPandas()

pivot_sen = senioridade_ano.pivot(index="ano_pesquisa", columns="nivel_senioridade", values="total").fillna(0)
ordem = [c for c in ["Júnior","Pleno","Sênior","Especialista/Staff+"] if c in pivot_sen.columns]
pivot_sen = pivot_sen.reindex(columns=ordem)
pivot_sen_pct = pivot_sen.div(pivot_sen.sum(axis=1), axis=0)*100

plot_100_stacked(
    pivot_sen_pct,
    title="A composição da senioridade mudou ao longo das edições da pesquisa",
    subtitle="Percentual de profissionais por nível de senioridade em cada ano",
    filename="02_evolucao_senioridade_pct.png",
    colors=[PALETTE["neutral"], "#7AA6C2", PALETTE["primary"], PALETTE["accent"]],
    legend_title="Senioridade"
)

display(pivot_sen.assign(Total=pivot_sen.sum(axis=1)))

# 2. Quais perfis profissionais são mais valorizados pelo mercado?

A análise é dividida em dois indicadores:

- **Volume:** quantos profissionais de cada cargo estão nas faixas altas.
- **Concentração:** qual percentual daquele cargo está nas faixas altas.

In [ ]:
# 2A — Volume absoluto em faixas salariais altas
salario_volume = spark.sql('''
    SELECT cargo_atual, COUNT(*) AS profissionais_faixa_alta
    FROM state_of_data
    WHERE ano_pesquisa = '2025'
      AND cargo_atual IS NOT NULL
      AND TRIM(cargo_atual) != ''
      AND (
          faixa_salarial LIKE '%16.001%'
          OR LOWER(faixa_salarial) LIKE '%acima de%'
      )
    GROUP BY cargo_atual
    ORDER BY profissionais_faixa_alta DESC
    LIMIT 10
''').toPandas()

total_faixa_alta = salario_volume["profissionais_faixa_alta"].sum()

plot_hbar(
    salario_volume,
    category="cargo_atual",
    value="profissionais_faixa_alta",
    title="Cargos com maior volume de profissionais nas faixas salariais superiores",
    subtitle="Profissionais com remuneração nas faixas a partir de R$ 16 mil | 2025",
    filename="03_volume_faixa_salarial_alta.png",
    highlight_top=2,
    color=PALETTE["accent"],
    wrap=42
)

print(f"{fmt_int(total_faixa_alta)} profissionais distribuídos entre os 10 cargos líderes.")

In [ ]:
# 2B — Concentração percentual por cargo
salario_concentracao = spark.sql('''
    WITH base AS (
        SELECT
            cargo_atual,
            COUNT(*) AS total_cargo,
            SUM(
                CASE
                    WHEN faixa_salarial LIKE '%16.001%'
                      OR LOWER(faixa_salarial) LIKE '%acima de%'
                    THEN 1 ELSE 0
                END
            ) AS alta
        FROM state_of_data
        WHERE ano_pesquisa = '2025'
          AND cargo_atual IS NOT NULL
          AND TRIM(cargo_atual) != ''
        GROUP BY cargo_atual
    )
    SELECT
        cargo_atual,
        total_cargo,
        alta,
        ROUND(100.0 * alta / total_cargo, 2) AS pct_faixa_alta
    FROM base
    WHERE total_cargo >= 10
      AND alta > 0
    ORDER BY pct_faixa_alta DESC, alta DESC
    LIMIT 10
''').toPandas()

plot_hbar(
    salario_concentracao,
    category="cargo_atual",
    value="pct_faixa_alta",
    title="Cargos com maior concentração proporcional nas faixas salariais superiores",
    subtitle="Percentual de profissionais de cada cargo em faixas a partir de R$ 16 mil | filtro mínimo: 10 respondentes por cargo",
    filename="04_concentracao_faixa_salarial_alta_pct.png",
    highlight_top=3,
    color=PALETTE["accent2"],
    wrap=42,
    value_fmt=lambda v: fmt_pct(v,1)
)

display(salario_concentracao)

# 3. Qual o cenário de diversidade de gênero nas carreiras de Dados?


In [ ]:
# 3A — Gênero por ano
genero_ano = spark.sql('''
    SELECT ano_pesquisa, genero, COUNT(*) AS total
    FROM state_of_data
    WHERE genero IS NOT NULL
      AND TRIM(genero) != ''
    GROUP BY ano_pesquisa, genero
''').toPandas()

pivot_genero = genero_ano.pivot(index="ano_pesquisa", columns="genero", values="total").fillna(0)
pivot_genero_pct = pivot_genero.div(pivot_genero.sum(axis=1), axis=0)*100

cores_genero = []
for c in pivot_genero_pct.columns:
    if c == "Feminino":
        cores_genero.append(PALETTE["female"])
    elif c == "Masculino":
        cores_genero.append(PALETTE["male"])
    else:
        cores_genero.append(PALETTE["neutral"])

plot_100_stacked(
    pivot_genero_pct,
    title="Representatividade de gênero ao longo das edições da pesquisa",
    subtitle="Distribuição percentual por ano",
    filename="05_genero_evolucao_pct.png",
    colors=cores_genero,
    legend_title="Gênero"
)

In [ ]:
# 3B — Gênero dentro da senioridade
genero_senioridade = spark.sql('''
    SELECT genero, nivel_senioridade, COUNT(*) AS total
    FROM state_of_data
    WHERE ano_pesquisa = '2025'
      AND genero IN ('Masculino','Feminino')
      AND nivel_senioridade IS NOT NULL
      AND TRIM(nivel_senioridade) != ''
    GROUP BY genero, nivel_senioridade
''').toPandas()

pivot_gs = genero_senioridade.pivot(index="nivel_senioridade", columns="genero", values="total").fillna(0)
ordem = [c for c in ["Júnior","Pleno","Sênior","Especialista/Staff+"] if c in pivot_gs.index]
pivot_gs = pivot_gs.reindex(ordem)
pivot_gs_pct = pivot_gs.div(pivot_gs.sum(axis=1), axis=0)*100

plot_100_stacked(
    pivot_gs_pct,
    title="A representatividade de gênero varia entre os níveis de senioridade",
    subtitle="Percentual dentro de cada nível | 2025",
    filename="06_genero_por_senioridade_pct.png",
    colors=[PALETTE["female"], PALETTE["male"]],
    legend_title="Gênero",
    horizontal=True
)

# 4. Quais tecnologias apresentam maior adoção?

Como os campos são de múltipla seleção, as visualizações mostram:
- ranking em 2025;
- percentual de respondentes válidos que selecionaram cada tecnologia;
- evolução temporal em percentual, evitando distorção por amostras diferentes.

In [ ]:
# Explode de linguagens
linguagens = spark.sql('''
    SELECT ano_pesquisa, linguagens_utilizadas
    FROM state_of_data
    WHERE linguagens_utilizadas IS NOT NULL
      AND TRIM(linguagens_utilizadas) != ''
''')

linguagens_explodido = (
    linguagens
    .withColumn("linguagem", F.explode(F.split(F.col("linguagens_utilizadas"), ",\\s*")))
    .withColumn("linguagem", F.trim(F.col("linguagem")))
    .filter(~F.col("linguagem").contains("Não utilizo"))
)

den_lang_2025 = (
    linguagens
    .filter(F.col("ano_pesquisa")=="2025")
    .count()
)

top_linguagens_2025 = (
    linguagens_explodido
    .filter(F.col("ano_pesquisa")=="2025")
    .groupBy("linguagem").count()
    .withColumnRenamed("count","total")
    .withColumn("pct_respondentes", F.round(F.lit(100.0)*F.col("total")/F.lit(den_lang_2025),2))
    .orderBy(F.desc("pct_respondentes"), F.desc("total"))
    .limit(10)
    .toPandas()
)

plot_hbar(
    top_linguagens_2025,
    category="linguagem",
    value="pct_respondentes",
    title="Tecnologias de programação mais utilizadas no trabalho",
    subtitle=f"Percentual de respondentes válidos que selecionaram cada linguagem | 2025 | Base: {fmt_int(den_lang_2025)}",
    filename="07_top_linguagens_2025_pct.png",
    highlight_top=3,
    color=PALETTE["accent2"],
    value_fmt=lambda v: fmt_pct(v,1),
    wrap=30
)

In [ ]:
# Explode de cloud
cloud = spark.sql('''
    SELECT ano_pesquisa, cloud_utilizada
    FROM state_of_data
    WHERE cloud_utilizada IS NOT NULL
      AND TRIM(cloud_utilizada) != ''
''')

cloud_explodido = (
    cloud
    .withColumn("cloud", F.explode(F.split(F.col("cloud_utilizada"), ",\\s*")))
    .withColumn("cloud", F.trim(F.col("cloud")))
)

den_cloud_2025 = cloud.filter(F.col("ano_pesquisa")=="2025").count()

top_cloud_2025 = (
    cloud_explodido
    .filter(F.col("ano_pesquisa")=="2025")
    .groupBy("cloud").count()
    .withColumnRenamed("count","total")
    .withColumn("pct_respondentes", F.round(F.lit(100.0)*F.col("total")/F.lit(den_cloud_2025),2))
    .orderBy(F.desc("pct_respondentes"), F.desc("total"))
    .limit(10)
    .toPandas()
)

plot_hbar(
    top_cloud_2025,
    category="cloud",
    value="pct_respondentes",
    title="Provedores de Cloud com maior adoção entre profissionais de Dados",
    subtitle=f"Percentual de respondentes válidos que selecionaram cada provedor | 2025 | Base: {fmt_int(den_cloud_2025)}",
    filename="08_top_cloud_2025_pct.png",
    highlight_top=3,
    color=PALETTE["accent"],
    value_fmt=lambda v: fmt_pct(v,1),
    wrap=34
)

In [ ]:
# Evolução Python, SQL e R em percentual
den_lang_ano = (
    linguagens.groupBy("ano_pesquisa").count()
    .withColumnRenamed("count","base")
)

evolucao_linguagens = (
    linguagens_explodido
    .filter(F.col("linguagem").isin("Python","SQL","R"))
    .groupBy("ano_pesquisa","linguagem").count()
    .join(den_lang_ano, "ano_pesquisa")
    .withColumn("pct", 100*F.col("count")/F.col("base"))
    .select("ano_pesquisa","linguagem","pct")
    .toPandas()
)

pivot_lang = evolucao_linguagens.pivot(index="ano_pesquisa", columns="linguagem", values="pct").fillna(0)
pivot_lang = pivot_lang.reindex(columns=[c for c in ["Python","SQL","R"] if c in pivot_lang.columns])

fig, ax = plt.subplots(figsize=(12,7))
x = list(range(len(pivot_lang.index)))
colors = [PALETTE["primary"], PALETTE["accent"], PALETTE["accent2"]]
for col, color in zip(pivot_lang.columns, colors):
    ax.plot(x, pivot_lang[col], marker="o", linewidth=2.8, markersize=7, label=col, color=color)
    for xi, yi in zip(x, pivot_lang[col]):
        ax.text(xi, yi+1.2, fmt_pct(yi,0), ha="center", fontsize=8, color=color, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(pivot_lang.index.astype(str))
ax.set_ylim(0, max(100, pivot_lang.max().max()+12))
ax.yaxis.set_major_formatter(PercentFormatter(100))
base_axis(ax)
set_exec_title(
    ax,
    "A evolução das linguagens mais relevantes do ecossistema de Dados",
    "Percentual de respondentes válidos que selecionaram Python, SQL ou R"
)
ax.legend(frameon=False, ncol=3, loc="upper left")
save_exec(fig, "09_evolucao_linguagens_pct.png")

# 5. Qual o índice de adoção de Inteligência Artificial e seu impacto?

O gráfico principal usa a base de respostas válidas para medir adoção e evitar que “não respondeu” reduza artificialmente o indicador.

In [ ]:
# Base 5A — Adoção de IA por ano (respostas válidas)
adocao_ia_ano = spark.sql('''
    SELECT
        ano_pesquisa,
        CASE
            WHEN usa_ia_generativa_trabalho LIKE '%Não utilizo%' THEN 'Não usa'
            ELSE 'Usa'
        END AS status_uso_ia,
        COUNT(*) AS total
    FROM state_of_data
    WHERE usa_ia_generativa_trabalho IS NOT NULL
      AND TRIM(usa_ia_generativa_trabalho) != ''
    GROUP BY ano_pesquisa,
             CASE WHEN usa_ia_generativa_trabalho LIKE '%Não utilizo%' THEN 'Não usa' ELSE 'Usa' END
''').toPandas()

pivot_ia = adocao_ia_ano.pivot(index="ano_pesquisa", columns="status_uso_ia", values="total").fillna(0)
pivot_ia_pct = pivot_ia.div(pivot_ia.sum(axis=1), axis=0)*100

plot_100_stacked(
    pivot_ia_pct,
    title="A adoção de IA Generativa no trabalho ao longo dos anos",
    subtitle="Percentual entre respondentes que preencheram a pergunta",
    filename="10_adocao_ia_ano_pct.png",
    colors=["#D9A441", PALETTE["success"]] if set(pivot_ia_pct.columns)=={"Não usa","Usa"} else [PALETTE["neutral"], PALETTE["success"]],
    legend_title="Status"
)

if "Usa" in pivot_ia_pct.columns:
    print("Indicador executivo 2025:", fmt_pct(pivot_ia_pct.loc["2025","Usa"],1) if "2025" in pivot_ia_pct.index else "Ano 2025 indisponível")

In [ ]:
# Base 5B — IA por senioridade em 2025
adocao_ia_sen = spark.sql('''
    SELECT
        nivel_senioridade,
        CASE
            WHEN usa_ia_generativa_trabalho LIKE '%Não utilizo%' THEN 'Não usa'
            ELSE 'Usa'
        END AS status_uso_ia,
        COUNT(*) AS total
    FROM state_of_data
    WHERE ano_pesquisa='2025'
      AND nivel_senioridade IS NOT NULL
      AND TRIM(nivel_senioridade) != ''
      AND usa_ia_generativa_trabalho IS NOT NULL
      AND TRIM(usa_ia_generativa_trabalho) != ''
    GROUP BY nivel_senioridade,
             CASE WHEN usa_ia_generativa_trabalho LIKE '%Não utilizo%' THEN 'Não usa' ELSE 'Usa' END
''').toPandas()

pivot_ia_sen = adocao_ia_sen.pivot(index="nivel_senioridade", columns="status_uso_ia", values="total").fillna(0)
ordem = [c for c in ["Júnior","Pleno","Sênior","Especialista/Staff+"] if c in pivot_ia_sen.index]
pivot_ia_sen = pivot_ia_sen.reindex(ordem)
pivot_ia_sen_pct = pivot_ia_sen.div(pivot_ia_sen.sum(axis=1), axis=0)*100

plot_100_stacked(
    pivot_ia_sen_pct,
    title="A adoção de IA Generativa por nível de senioridade",
    subtitle="Percentual dentro de cada nível | 2025",
    filename="11_adocao_ia_senioridade_pct.png",
    colors=["#D9A441", PALETTE["success"]] if set(pivot_ia_sen_pct.columns)=={"Não usa","Usa"} else [PALETTE["neutral"], PALETTE["success"]],
    legend_title="Status",
    horizontal=True
)

# 6. Existem diferenças relevantes entre regiões e modelos de trabalho?

In [ ]:
# Base 6A — Regiões por ano
regiao_ano = spark.sql('''
    SELECT ano_pesquisa, regiao_onde_mora, COUNT(*) AS total
    FROM state_of_data
    WHERE regiao_onde_mora IS NOT NULL
      AND TRIM(regiao_onde_mora) != ''
    GROUP BY ano_pesquisa, regiao_onde_mora
''').toPandas()

pivot_regiao = regiao_ano.pivot(index="ano_pesquisa", columns="regiao_onde_mora", values="total").fillna(0)
pivot_regiao_pct = pivot_regiao.div(pivot_regiao.sum(axis=1), axis=0)*100

region_colors = sns.color_palette("Blues", n_colors=max(3, len(pivot_regiao_pct.columns))).as_hex()
plot_100_stacked(
    pivot_regiao_pct,
    title="Distribuição regional dos profissionais de Dados",
    subtitle="Participação percentual por ano",
    filename="12_regiao_evolucao_pct.png",
    colors=region_colors[:len(pivot_regiao_pct.columns)],
    legend_title="Região"
)

In [ ]:
# Base 6B — Modelo de trabalho (percentual)
modelo_trabalho = spark.sql('''
    SELECT ano_pesquisa, modelo_trabalho_atual, COUNT(*) AS total
    FROM state_of_data
    WHERE modelo_trabalho_atual IS NOT NULL
      AND TRIM(modelo_trabalho_atual) != ''
    GROUP BY ano_pesquisa, modelo_trabalho_atual
''').toPandas()

pivot_modelo = modelo_trabalho.pivot(index="ano_pesquisa", columns="modelo_trabalho_atual", values="total").fillna(0)
pivot_modelo_pct = pivot_modelo.div(pivot_modelo.sum(axis=1), axis=0)*100

model_colors = sns.color_palette("Purples", n_colors=max(3, len(pivot_modelo_pct.columns))).as_hex()
plot_100_stacked(
    pivot_modelo_pct,
    title="Como evoluiu o modelo de trabalho entre os profissionais de Dados",
    subtitle="Percentual por ano | 2023 pode não possuir respostas caso a pergunta não existisse",
    filename="13_modelo_trabalho_pct.png",
    colors=model_colors[:len(pivot_modelo_pct.columns)],
    legend_title="Modelo"
)

# 7. Fechamento executivo: oportunidades e desafios

In [ ]:
# Geração de resumo para ajudar nos insights
resumo = {}

if len(cargos_2025):
    resumo["cargo_lider"] = cargos_2025.iloc[0]["cargo_atual"]
    resumo["cargo_lider_total"] = int(cargos_2025.iloc[0]["total"])

if len(salario_volume):
    resumo["cargo_volume_alta"] = salario_volume.iloc[0]["cargo_atual"]
    resumo["volume_alta"] = int(salario_volume.iloc[0]["profissionais_faixa_alta"])

if len(salario_concentracao):
    resumo["cargo_concentracao_alta"] = salario_concentracao.iloc[0]["cargo_atual"]
    resumo["pct_concentracao_alta"] = float(salario_concentracao.iloc[0]["pct_faixa_alta"])

if "Usa" in pivot_ia_pct.columns and "2025" in pivot_ia_pct.index:
    resumo["ia_2025"] = float(pivot_ia_pct.loc["2025","Usa"])

print("RESUMO EXECUTIVO (preenchido com os dados da execução)\n")
for k,v in resumo.items():
    print(f"- {k}: {v}")

print("\nEstrutura recomendada para o slide final:")
print("1. Mercado: destaque o cargo líder e a composição de senioridade.")
print("2. Remuneração: diferencie volume de concentração nas faixas altas.")
print("3. Diversidade: cite apenas a tendência observada nos gráficos percentuais.")
print("4. Tecnologia: destaque liderança de linguagens/cloud e evolução temporal.")
print("5. IA: apresente o índice de adoção e a diferença entre senioridades, se houver.")
print("6. Estratégia: conecte os achados às decisões de contratação, retenção, capacitação e arquitetura de dados.")

# Exportação dos gráficos


In [ ]:
#Upload para pasta no S3
Descomente somente após definir bucket e prefixo.

import boto3
s3 = boto3.client("s3")
BUCKET = "tech-challenge-aws-data-hackers"
PREFIX = "gold/graficos_executivos"
for p in pngs:
    s3.upload_file(p, BUCKET, f"{PREFIX}/{Path(p).name}")
print(f"Arquivos enviados para s3://{BUCKET}/{PREFIX}/")